In [1]:
# now to downlaod a dataset that deals with dwellings for sa2 zones

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [2]:
# Download Data

Current_Path = Path.cwd()
Path_Root = Current_Path.parent.parent

Data_Path = Path_Root/"data"/"forecasting"
File_Path = Data_Path/"DwellingStock_1626.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)

Url = ("https://www.abs.gov.au/statistics/industry/"
       "building-and-construction/estimated-dwelling-stock/"
       "jun-quarter-2022/ASGS21_2016q2-2022q2.xlsx"
       )

Response = requests.get(
    Url, 
    timeout=60
)

Response.raise_for_status()

with open(File_Path, "wb") as File:
    File.write(Response.content)


print(f"Downloaded successfully to: {File_Path}")



Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/forecasting/DwellingStock_1626.xlsx


In [4]:
Excel_Data = pd.read_excel(File_Path, 
                           sheet_name="Table 1",
                           header=None
                           )
Csv_Unclean_Path = Data_Path/"DwellingStock_1626_Unclean.csv"
Excel_Data.to_csv(Csv_Unclean_Path)

In [7]:
# Now to clean the data

Unclean_Data = pd.read_csv(
    Csv_Unclean_Path,
    header=None,
    dtype=str
)


# Quarterly dates are stored in row 7
Quarter_Labels = (
    pd.to_datetime(
        Unclean_Data.iloc[7, 11:36],
        errors="coerce"
    )
    .dt.strftime("%Y-%m-%d")
    .tolist()
)

# Actual SA2 observations begin at row 10
#
# Column 9  = SA2 Code
# Column 10 = SA2 Name
# Columns 11:36 = Quarterly dwelling stock

Dwelling_Stock = Unclean_Data.iloc[
    10:,
    [9, 10] + list(range(11, 36))
].copy()

# Rename columns

Dwelling_Stock.columns = (
    ["SA2_Code", "SA2_Name"]
    + [
        f"Dwelling_Stock_{date}"
        for date in Quarter_Labels
    ]
)

# Clean SA2 columns

Dwelling_Stock["SA2_Code"] = (
    Dwelling_Stock["SA2_Code"]
    .str.strip()
)

Dwelling_Stock["SA2_Name"] = (
    Dwelling_Stock["SA2_Name"]
    .str.strip()
)


# Keep only genuine SA2 rows
Dwelling_Stock = Dwelling_Stock[
    Dwelling_Stock["SA2_Code"].str.fullmatch(
        r"\d+",
        na=False
    )
]


# --------------------------------------------------
# Convert dwelling stock to numeric
# --------------------------------------------------

for Column in Dwelling_Stock.columns[2:]:

    Dwelling_Stock[Column] = pd.to_numeric(
        Dwelling_Stock[Column],
        errors="coerce"
    ).astype("Int64")


# Reset index
Dwelling_Stock = Dwelling_Stock.reset_index(
    drop=True
)

In [10]:
Csv_Clean_Path = Data_Path/"DwellingStock1626_Clean.csv"
Dwelling_Stock.to_csv(Csv_Clean_Path)